# Will a UBC station still have an e-bike when I arrive?

Brock Commons shows one e-bike. You are ten minutes away. Does the station's identity, the time of day, or the way its inventory has been changing tell you enough to suggest a backup?

The unit of analysis is a **simulated walking decision** at a station reporting a positive e-bike count. The outcome is zero reported e-bikes or an explicit rental closure at the walking ETA. Multiple walks can overlap the same shortage. An unlock attempt is a different outcome and is unavailable in these data.

We audit the archive, replay one decision by hand, then compare progressively richer features on the same monthly folds. October 2026 was already used in the first pilot. We retain it as a retrospective comparison and reserve any claim about future riders for fresh observations.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/prepare_data.py").exists())
sys.path.insert(0, str(root / "notebooks"))
from availability_tools import (covered_times, forward_split, metrics, paired_precision_interval,
                                precision_within_stations, predict, station_decisions, warning_weights)

In [2]:
HORIZONS = (5, 10)
FRESH_MIN = 10
WARNING_RATE = 0.20
TIE_MARGIN = 0.01  # one percentage point of warning precision
TZ = "America/Vancouver"
TEST_MONTH = pd.Timestamp("2026-10-01", tz="UTC")
VALIDATION = pd.date_range(end=TEST_MONTH - pd.offsets.MonthBegin(1), periods=3, freq="MS")
print("Validation:", ", ".join(VALIDATION.strftime("%b %Y")), "| Retrospective test:", TEST_MONTH.strftime("%b %Y"))

Validation: Jul 2026, Aug 2026, Sep 2026 | Retrospective test: Oct 2026


## 1. What did the archive observe?

CityBikes saves station changes, chiefly changes in total bikes or free docks. A quiet station and a stopped collector can look alike in one station's history. We also load citywide observation times to distinguish long network outages. [Archive](https://data.citybik.es/) · [Collector](https://github.com/citybikes/analytics/blob/master/collect.py)

In [3]:
data = root / "data/processed"
inventory = pd.read_parquet(data / "ubc_inventory.parquet")
network = pd.DatetimeIndex(pd.read_parquet(data / "network_observations.parquet").observed_at)
end = TEST_MONTH + pd.offsets.MonthBegin(1)
inventory = inventory.loc[inventory.observed_at < end].sort_values(["station_id", "observed_at"]).copy()
network = network[network < end].sort_values().unique()
assert not inventory.duplicated(["station_id", "observed_at"]).any()
assert inventory.observed_at.isin(network).all()
print(f"{len(inventory):,} station changes at {inventory.station_id.nunique()} hubs")
display(inventory[["station_name", "observed_at", "ebikes", "classic_bikes", "total_bikes", "free_docks"]].head(4))

189,751 station changes at 17 hubs


,station_name,observed_at,ebikes,classic_bikes,total_bikes,free_docks
0,Place Vanier,2025-05-14 19:39:15+00:00,0,0,0,0
1,Place Vanier,2025-07-30 20:20:59+00:00,0,0,0,15
2,Place Vanier,2025-07-30 21:09:00+00:00,1,0,1,14
3,Place Vanier,2025-07-31 00:08:59+00:00,0,0,0,15


In [4]:
development = inventory.loc[inventory.observed_at < VALIDATION[0]]
coverage = development.groupby("station_name").observed_at.agg(["count", "min", "max"])
display(coverage.sort_values("count").iloc[[0, 1, -2, -1]].rename(columns={"count": "reports"}))
network_gaps = network.to_series().diff().nlargest(3)
display(pd.DataFrame({"last report": network_gaps.index - network_gaps, "next report": network_gaps.index,
                      "gap": network_gaps.values}).reset_index(drop=True))

,reports,min,max
station_name,,,
Binning & Wesbrook,2433,2025-11-17 21:56:42+00:00,2026-06-30 22:56:48+00:00
Acadia Park,3560,2025-05-14 19:39:15+00:00,2026-06-30 22:39:48+00:00
Orchard Commons,12925,2024-11-01 15:13:32+00:00,2026-06-01 13:10:25+00:00
Wesbrook Village - Berton & Shrum,13471,2024-11-01 15:13:32+00:00,2026-06-30 23:42:48+00:00


,last report,next report,gap
0,2025-05-21 05:36:15+00:00,2025-07-06 10:50:51+00:00,46 days 05:14:36
1,2025-02-21 08:50:21+00:00,2025-02-26 15:57:34+00:00,5 days 07:07:13
2,2025-01-17 14:25:34+00:00,2025-01-21 11:27:33+00:00,3 days 21:01:59


The largest gap lasts over 46 days. Forward-filling through it would manufacture thousands of walks. We will require citywide activity bracketing every usable interval. Station report counts also differ, so rows of raw changes are a poor stand-in for random walk start times.

In [5]:
bad_count = inventory.ebikes.lt(0) | inventory.ebikes.gt(inventory.total_bikes)
type_disagreement = (inventory.ebikes + inventory.classic_bikes).ne(inventory.total_bikes)
display(pd.Series({
    "missing e-bike count": inventory.ebikes.isna().sum(),
    "e-bike count negative or exceeds total": bad_count.sum(),
    "known type counts sum to a different total": type_disagreement.sum(),
    "explicit rental closures": inventory.is_renting.eq(False).sum(),
}, name="archive rows").to_frame())

,archive rows
missing e-bike count,544
e-bike count negative or exceeds total,30
known type counts sum to a different total,109
explicit rental closures,319


An unknown e-bike count can never establish an empty station. If the e-bike count itself is impossible, mark it missing. This changes 30 reports in this archive. A disagreement involving the classic count alone leaves a possible reported e-bike count usable. Retain explicit closures so a rider arriving at a closed dock is counted as unable to rent.

In [6]:
inventory.loc[bad_count.fillna(False), "ebikes"] = pd.NA
inventory.loc[inventory.total_bikes.lt(0).fillna(False), "total_bikes"] = pd.NA
assert inventory.ebikes.dropna().ge(0).all()

## 2. Reconstruct one walk before generating the cohort

Sample the wall clock every five minutes. Require citywide archive gaps of at most ten minutes across the 15-minute history window and the following ten-minute walk. At departure, require a station report at most ten minutes old. At arrival, use the latest archived state by the deadline, including an unchanged state. Requiring a new station report during the walk would preferentially select busy stations.

In [7]:
grid = covered_times(network, interval_min=5, history_min=15, arrival_min=max(HORIZONS), max_gap_min=10)
brock = inventory.loc[inventory.station_id.eq("2131")]
brock_walks = station_decisions(brock, grid, HORIZONS, FRESH_MIN)
# Pick a training-period example where the last e-bike disappears before the longer walk ends.
case = brock_walks.loc[(brock_walks.time < VALIDATION[0]) & brock_walks.ebikes.eq(1)
                       & brock_walks.arrival_ebikes_10.eq(0)].iloc[0]
local_case = pd.Series({"walk starts": case.time.tz_convert(TZ), "e-bikes at start": case.ebikes,
                        "5-min arrival count": case.arrival_ebikes_5, "10-min arrival count": case.arrival_ebikes_10})
display(local_case.to_frame("Brock Commons example"))

,Brock Commons example
walk starts,2024-11-02 16:50:00-07:00
e-bikes at start,1
5-min arrival count,0.0
10-min arrival count,0.0


In [8]:
window = brock.observed_at.between(case.time - pd.Timedelta(minutes=10), case.time + pd.Timedelta(minutes=10))
display(brock.loc[window, ["observed_at", "ebikes", "total_bikes", "is_renting"]]
        .assign(observed_at=lambda x: x.observed_at.dt.tz_convert(TZ)))
for h in HORIZONS:
    eta = case.time + pd.Timedelta(minutes=h)
    assert case[f"arrival_report_{h}"] <= eta
    assert brock.loc[brock.observed_at <= eta, "ebikes"].iloc[-1] == case[f"arrival_ebikes_{h}"]

,observed_at,ebikes,total_bikes,is_renting
60590,2024-11-02 16:49:55-07:00,1,6,True
60591,2024-11-02 16:52:54-07:00,0,5,True


At 16:50 on November 2, 2024, the last Brock report was five seconds old and showed one e-bike. The next report, at 16:52:54, showed zero. Both walking horizons therefore have an empty arrival. Now apply the same lookup to every station. At either ETA, a closure establishes unavailability even if the bike count is missing. For an open station with an unknown count the outcome stays unknown. We compare both horizons on the same complete decisions.

In [9]:
walks = pd.concat([station_decisions(group, grid, HORIZONS, FRESH_MIN)
                   for _, group in inventory.groupby("station_id")], ignore_index=True)
for h in HORIZONS:
    closed = walks[f"arrival_closed_{h}"]
    count = walks[f"arrival_ebikes_{h}"]
    walks[f"empty_{h}"] = (closed | count.eq(0)).where(closed | count.notna())
    assert (walks[f"arrival_report_{h}"] <= walks.time + pd.Timedelta(minutes=h)).all()
before = len(walks)
walks = walks.dropna(subset=["empty_5", "empty_10"]).sort_values("time").reset_index(drop=True)
print(f"{before:,} eligible decisions; {before - len(walks):,} missing a paired outcome; {len(walks):,} retained.")
display(pd.Series({f"{h}-min walk": (walks[f"arrival_report_{h}"] > walks.observed_at).mean()
                   for h in HORIZONS}, name="fraction with a newer station report").round(3).to_frame())

236,275 eligible decisions; 8 missing a paired outcome; 236,267 retained.


,fraction with a newer station report
5-min walk,0.161
10-min walk,0.276


Most walks reuse the station's last report for the arrival label. That is a limitation of the feed. A five-minute model can learn which reported states tend to change, while failures to unlock an apparently available bike remain invisible.

In [10]:
early = walks.loc[walks.time + pd.Timedelta(minutes=max(HORIZONS)) < VALIDATION[0]]
by_count = early.groupby(early.ebikes.clip(upper=4)).agg(
    decisions=("time", "size"), empty_5=("empty_5", "mean"), empty_10=("empty_10", "mean"))
by_count.index = ["1", "2", "3", "4+"]
display(by_count.astype({"empty_5": float, "empty_10": float}).round(4))

,decisions,empty_5,empty_10
1,52585,0.0534,0.0903
2,40273,0.0047,0.0121
3,28076,0.0006,0.0020
4+,56918,0.0001,0.0002


Starting inventory is already a strong predictor. The useful comparison is how many extra empty arrivals we can flag at the same warning volume. Walks starting with exactly one bike will get a separate evaluation because count alone gives all of them the same score.

## 3. Establish the comparison before fitting models

Train on earlier months and move the evaluation window forward one month at a time. Discard training decisions whose ten-minute arrival touches the validation period. The same rule applies at each evaluation window's far boundary.

In [11]:
for month in [*VALIDATION, TEST_MONTH]:
    days = network[(network >= month) & (network < month + pd.offsets.MonthBegin(1))].normalize().unique()
    assert len(days) >= 0.9 * month.days_in_month and days.min() == month
    assert days.max() == month + pd.offsets.MonthBegin(1) - pd.Timedelta(days=1)
folds = [forward_split(walks, m, max(HORIZONS)) for m in VALIDATION]
for train, val in folds:
    assert train.time.max() + pd.Timedelta(minutes=max(HORIZONS)) < val.time.min()
display(pd.DataFrame([{"validation": m.strftime("%Y-%m"), "training walks": len(t),
                       "validation walks": len(v), "latest training start": t.time.max()}
                      for m, (t, v) in zip(VALIDATION, folds)]))
fold_indices = [(t.index, v.index) for t, v in folds]
del folds

,validation,training walks,validation walks,latest training start
0,2026-07,177852,12446,2026-06-30 23:45:00+00:00
1,2026-08,190307,11193,2026-07-31 23:45:00+00:00
2,2026-09,201507,15801,2026-08-31 23:35:00+00:00


Issue 20 warnings per 100 eligible decisions. **Precision** counts empty arrivals per warning. **Recall** counts the fraction of empty arrivals that receive a warning. Average precision evaluates ranking across budgets, and Brier score evaluates the probability estimates. Give tied scores equal fractional credit at the warning cutoff. When five walks get the same risk, one warning gives each walk one fifth of the credit.

In [12]:
assert np.allclose(warning_weights([0.5] * 5, WARNING_RATE), [0.2] * 5)
display(pd.Series(metrics([1, 0, 0, 0, 0], [0.5] * 5, WARNING_RATE)).round(3).to_frame("tied example"))

,tied example
walks,5.00
empty_arrivals,1.00
empty_rate,0.20
precision,0.20
recall,0.20
average_precision,0.20
brier,0.25


## 4. Compare raw inputs, then engineer features

Each experiment below uses the same three validation folds and same two walking times. Logistic regression standardizes continuous fields and one-hot encodes the hub. A shallow boosted tree uses the same fields and imputation. All transformations fit on the training period. The reusable runner handles repeated fits and scoring. Each experiment cell specifies its inputs and immediately displays its results.

In [13]:
def pipeline(estimator, fields):
    numbers = [f for f in fields if f != "station_id"]
    numeric = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())
    transforms = [("numbers", numeric, numbers)]
    if "station_id" in fields:
        transforms.append(("hub", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["station_id"]))
    return make_pipeline(ColumnTransformer(transforms), estimator)

logistic = LogisticRegression(max_iter=400)
tree = HistGradientBoostingClassifier(max_depth=3, max_iter=80, learning_rate=.07, min_samples_leaf=40,
                                      l2_regularization=1, early_stopping=False, random_state=42)

In [14]:
experiments, specifications = {}, {}

def evaluate(name, estimator, fields):
    model = None if estimator is None else pipeline(estimator, fields)
    records = []
    for month, (train_idx, val_idx) in zip(VALIDATION, fold_indices):
        used = fields + ["empty_5", "empty_10"]
        train, val = walks.loc[train_idx, used], walks.loc[val_idx, used]
        one = walks.loc[val_idx, "ebikes"].eq(1).to_numpy(dtype=bool)
        hubs = walks.loc[val_idx, "station_id"].to_numpy()[one]
        for h in HORIZONS:
            score = predict(model, fields, train, val, h)
            y = val[f"empty_{h}"].to_numpy(dtype=int)
            row = metrics(y, score, WARNING_RATE)
            row["same_hub_one_bike"] = precision_within_stations(y[one], score[one], hubs, WARNING_RATE)
            records.append(dict(month=month.strftime("%Y-%m"), horizon=h, **row))
    specifications[name] = (model, fields)
    experiments[name] = pd.DataFrame(records)

def compare(*names):
    rows = pd.concat({name: experiments[name] for name in names}, names=["experiment"])
    by_horizon = rows.groupby(["experiment", "horizon"]).precision.mean()
    return (100 * by_horizon.unstack()).reindex(names).round(2).rename(columns={5: "5 min", 10: "10 min"})

### How far can raw counts get us?

Start with the historical empty-arrival rate for each e-bike count, smoothed toward the training mean. Pool counts of five or more. Then use exactly one input in logistic regression to see if changing the estimator makes a difference by itself.

In [15]:
counts = ["ebikes", "total_bikes"]
hub = ["station_id"]
evaluate("Count baseline", None, ["ebikes"])
evaluate("E-bike count", logistic, ["ebikes"])
compare("Count baseline", "E-bike count")

horizon,5 min,10 min
experiment,,
Count baseline,8.47,14.6
E-bike count,8.47,14.6


Both produce the same warning precision. Add the total number of bikes, then the station ID, keeping logistic regression fixed. This isolates the contribution of knowing which dock the rider is walking to.

In [16]:
evaluate("Both counts", logistic, counts)
evaluate("Counts + hub", logistic, counts + hub)
compare("E-bike count", "Both counts", "Counts + hub")

horizon,5 min,10 min
experiment,,
E-bike count,8.47,14.60
Both counts,8.83,15.63
Counts + hub,11.00,19.20


At ten minutes, precision rises from 15.63% with both counts to 19.20% with the hub. A station identifier costs almost nothing at prediction time. Keep it as a candidate and test whether the remaining raw station fields earn their place.

In [17]:
snapshot = counts + ["classic_bikes", "free_docks", "capacity"]
evaluate("Full snapshot + hub", logistic, snapshot + hub)
compare("Counts + hub", "Full snapshot + hub")

horizon,5 min,10 min
experiment,,
Counts + hub,11.00,19.20
Full snapshot + hub,11.01,19.23


Classic-bike count, free docks and capacity together move ten-minute precision by about 0.03 percentage points. Most of the usable inventory signal is already present in the two counts. The next comparison adds local time alone, then adds it alongside hub identity.

The hub accounts for most of the gain so far. Can a model get similar performance using only the hub and the advertised e-bike count? Drop total bikes explicitly before carrying it into every later experiment.

In [18]:
evaluate("E-bike count + hub", logistic, ["ebikes", "station_id"])
compare("E-bike count", "E-bike count + hub", "Counts + hub")

horizon,5 min,10 min
experiment,,
E-bike count,8.47,14.60
E-bike count + hub,10.89,18.73
Counts + hub,11.00,19.20


### Does a Vancouver clock add anything?

A model reading the hour as a single number would treat 23:59 and 00:01 as far apart. Encode the time of day on a circle and mark weekends. Inspect the resulting values before fitting.

In [19]:
local = walks.time.dt.tz_convert(TZ)
angle = 2 * np.pi * (local.dt.hour + local.dt.minute / 60) / 24
walks["hour_sin"] = np.sin(angle)
walks["hour_cos"] = np.cos(angle)
walks["weekend"] = (local.dt.dayofweek >= 5).astype(int)
clock = ["hour_sin", "hour_cos", "weekend"]
display(walks.assign(local_time=local).loc[:, ["local_time", *clock]].iloc[:3].round(3))

,local_time,hour_sin,hour_cos,weekend
0,2024-11-01 08:45:00-07:00,0.752,-0.659,0
1,2024-11-01 08:50:00-07:00,0.737,-0.676,0
2,2024-11-01 08:50:00-07:00,0.737,-0.676,0


In [20]:
evaluate("Counts + clock", logistic, counts + clock)
evaluate("Counts + hub + clock", logistic, counts + hub + clock)
compare("Both counts", "Counts + clock", "Counts + hub", "Counts + hub + clock")

horizon,5 min,10 min
experiment,,
Both counts,8.83,15.63
Counts + clock,9.86,16.87
Counts + hub,11.00,19.20
Counts + hub + clock,11.52,19.50


The clock helps more when the station is unknown. Once we know the hub, the ten-minute gain from adding the clock is about 0.30 percentage points. Can a signal measured just before departure do better?

### Does recent inventory movement distinguish two one-bike walks?

Use only information available by departure. A fall from three e-bikes to one contributes a net change of −2. It could reflect rentals, staff moves, or both. Report age measures the time since any station report. An unavailable prior reading stays missing and gets an imputation indicator during fitting.

In [21]:
for lag in (5, 15):
    walks[f"net_change_{lag}"] = walks.ebikes - walks[f"ebikes_{lag}m_ago"]
trend = ["net_change_5", "net_change_15"]
reporting = ["reports_15", "age_min"]
history = trend + reporting
assert walks.age_min.between(0, FRESH_MIN).all()
train_idx = fold_indices[0][0]
display(walks.loc[train_idx, history].isna().mean().rename("missing fraction in first training fold").to_frame().round(3))

,missing fraction in first training fold
net_change_5,0.405
net_change_15,0.770
reports_15,0.000
age_min,0.000


In [22]:
one_left = walks.loc[train_idx].query("ebikes == 1").copy()
change = one_left.net_change_5
one_left["prior five minutes"] = np.select(
    [change.isna(), change.lt(0).fillna(False), change.gt(0).fillna(False)],
    ["unknown", "net decrease", "net increase"], default="unchanged")
display(one_left.groupby("prior five minutes").agg(walks=("time", "size"),
        empty_in_10=("empty_10", "mean")).astype({"empty_in_10": float}).round(3))

,walks,empty_in_10
prior five minutes,,
net decrease,2794,0.109
net increase,1751,0.138
unchanged,26305,0.083
unknown,21735,0.093


In [23]:
evaluate("Counts + hub + history", logistic, counts + hub + history)
evaluate("Context + net changes", logistic, counts + hub + clock + trend)
evaluate("Context + reporting", logistic, counts + hub + clock + reporting)
compare("Counts + hub", "Counts + hub + history", "Counts + hub + clock",
        "Context + net changes", "Context + reporting")

horizon,5 min,10 min
experiment,,
Counts + hub,11.00,19.20
Counts + hub + history,11.04,19.37
Counts + hub + clock,11.52,19.50
Context + net changes,11.42,19.47
Context + reporting,11.46,19.53


In the early one-bike sample, the ten-minute empty rate was 8.3% after an unchanged count, 10.9% after a net decrease, and 13.8% after a net increase. A simple “bikes are leaving fast” story misses the last group. Some hubs and times may generate both more turnover and more shortages. Test both history groups together. Then restore the extra raw fields to check whether discarding them early concealed a useful combination.

In [24]:
context = counts + hub + clock
evaluate("Context + history", logistic, context + history)
evaluate("Full snapshot + clock + history", logistic, snapshot + hub + clock + history)
compare("Counts + hub + clock", "Context + net changes", "Context + reporting",
        "Context + history", "Full snapshot + clock + history")

horizon,5 min,10 min
experiment,,
Counts + hub + clock,11.52,19.50
Context + net changes,11.42,19.47
Context + reporting,11.46,19.53
Context + history,11.44,19.68
Full snapshot + clock + history,11.54,19.69


Net changes alone give 19.47% ten-minute warning precision, compared with 19.50% for the clock-and-hub model. Reporting activity alone gives 19.53%. Combining them reaches 19.68%. These are small differences across all walks. The later one-bike comparison tests whether averaging across the whole campus obscures a gain for someone repeatedly using the same station.

### Has the station been sitting at this count?

A five-minute net change cannot distinguish a station that just reached its last bike from one that has held it across several reports. Measure the length of the *observed* run at the same e-bike count. Restart the run after a gap over ten minutes, a change in rental status, or an unknown count. Cap the duration at thirty minutes so a sparse sequence cannot imply hours of persistence.

In [25]:
by_hub = inventory.groupby("station_id", sort=False)
same_count = inventory.ebikes.eq(by_hub.ebikes.shift()).fillna(False)
same_status = inventory.is_renting.eq(by_hub.is_renting.shift()).fillna(False)
long_gap = by_hub.observed_at.diff().gt(pd.Timedelta(minutes=FRESH_MIN))
restart = ~same_count | ~same_status | long_gap
run_start = inventory.observed_at.where(restart).groupby(inventory.station_id).ffill()
runs = inventory[["station_id", "observed_at"]].assign(run_start=run_start)
matched = walks[["station_id", "observed_at"]].merge(runs, how="left", validate="many_to_one")
assert matched.run_start.notna().all() and matched.observed_at.equals(walks.observed_at)
walks["at_count_min"] = ((walks.time - matched.run_start).dt.total_seconds() / 60).clip(upper=30)
assert walks.at_count_min.ge(0).all()
display(walks.loc[train_idx, ["age_min", "at_count_min"]].quantile([.25, .5, .75, .95]).round(1))

,age_min,at_count_min
0.25,2.2,2.3
0.50,4.3,4.7
0.75,7.2,7.4
0.95,9.3,9.6


In [26]:
evaluate("Context + time at count", logistic, context + ["at_count_min"])
evaluate("Context + history + time at count", logistic, context + history + ["at_count_min"])
compare("Counts + hub + clock", "Context + time at count", "Context + history",
        "Context + history + time at count")

horizon,5 min,10 min
experiment,,
Counts + hub + clock,11.52,19.50
Context + time at count,11.36,19.35
Context + history,11.44,19.68
Context + history + time at count,11.51,19.60


Adding observed time at count takes the ten-minute logistic model from 19.50% to 19.35% on its own, and from 19.68% to 19.60% when history is already included. The extra feature did not improve the campus-wide warning metric in these folds.

### Would a nonlinear model change the feature decision?

The best combination so far moves ten-minute precision by less than a point over counts and hub alone. Logistic regression assumes each feature has a linear effect on log-odds. A shallow tree can, for example, use a recent drop at a one-bike station differently from the same drop at a well-stocked station. First give it exactly the same raw fields.

In [27]:
def compare_tree(name):
    fields = specifications[name][1]
    evaluate("Tree · " + name, tree, fields)
    return compare(name, "Tree · " + name)

display(compare_tree("E-bike count"))
display(compare_tree("Both counts"))
display(compare_tree("Counts + hub"))
display(compare_tree("Full snapshot + hub"))

horizon,5 min,10 min
experiment,,
E-bike count,8.47,14.6
Tree · E-bike count,8.47,14.6


horizon,5 min,10 min
experiment,,
Both counts,8.83,15.63
Tree · Both counts,8.51,14.66


horizon,5 min,10 min
experiment,,
Counts + hub,11.00,19.20
Tree · Counts + hub,10.66,18.62


horizon,5 min,10 min
experiment,,
Full snapshot + hub,11.01,19.23
Tree · Full snapshot + hub,10.77,18.75


Now compare time and inventory history using the same tree settings. Keep the original validation folds, feature definitions, and warning budget fixed.

In [28]:
display(compare_tree("E-bike count + hub"))
display(compare_tree("Counts + clock"))
display(compare_tree("Counts + hub + clock"))
display(compare_tree("Counts + hub + history"))

horizon,5 min,10 min
experiment,,
E-bike count + hub,10.89,18.73
Tree · E-bike count + hub,10.58,18.75


horizon,5 min,10 min
experiment,,
Counts + clock,9.86,16.87
Tree · Counts + clock,9.82,17.18


horizon,5 min,10 min
experiment,,
Counts + hub + clock,11.52,19.50
Tree · Counts + hub + clock,11.23,18.76


horizon,5 min,10 min
experiment,,
Counts + hub + history,11.04,19.37
Tree · Counts + hub + history,10.92,18.54


In [29]:
display(compare_tree("Context + history"))
display(compare_tree("Full snapshot + clock + history"))

horizon,5 min,10 min
experiment,,
Context + history,11.44,19.68
Tree · Context + history,11.12,19.30


horizon,5 min,10 min
experiment,,
Full snapshot + clock + history,11.54,19.69
Tree · Full snapshot + clock + history,11.34,19.34


The duration of an unchanged count may interact with the number of bikes left. Give the tree that feature as well and compare it directly with the corresponding logistic fits.

In [30]:
display(compare_tree("Context + time at count"))
display(compare_tree("Context + history + time at count"))

horizon,5 min,10 min
experiment,,
Context + time at count,11.36,19.35
Tree · Context + time at count,11.00,19.28


horizon,5 min,10 min
experiment,,
Context + history + time at count,11.51,19.60
Tree · Context + history + time at count,10.95,19.29


## 5. Choose a model and inspect the trade-off

Rank by **mean warning precision across months**, with each month weighted equally. As in the first pilot, use a simpler configuration if it is within one percentage point of the best score for that horizon. For simplicity we prefer a count lookup first, then logistic models using fewer feature groups, then trees. The complete ranking is visible below. No October scores enter this choice.

In [31]:
# Explicit feature-cost order for near ties. Raw hub ID precedes engineered time and history.
preference = ["Count baseline", "E-bike count", "Both counts", "E-bike count + hub", "Counts + hub", "Full snapshot + hub",
              "Counts + clock", "Counts + hub + clock", "Counts + hub + history", "Context + net changes",
              "Context + reporting", "Context + time at count", "Context + history",
              "Context + history + time at count", "Full snapshot + clock + history"]
preference += ["Tree · " + name for name in preference if "Tree · " + name in experiments]
assert set(preference) == set(experiments)
all_scores = pd.concat(experiments, names=["experiment"]).reset_index(level=0)
means = all_scores.groupby(["experiment", "horizon"]).mean(numeric_only=True)
precision = means.precision.unstack().reindex(preference)
selected = {h: precision.index[precision[h] >= precision[h].max() - TIE_MARGIN][0] for h in HORIZONS}
display((100 * precision).round(2).rename(columns={5: "5-min precision (%)", 10: "10-min precision (%)"}))
print("Selected from validation:", selected)

horizon,5-min precision (%),10-min precision (%)
experiment,,
Count baseline,8.47,14.60
E-bike count,8.47,14.60
Both counts,8.83,15.63
E-bike count + hub,10.89,18.73
Counts + hub,11.00,19.20
Full snapshot + hub,11.01,19.23
Counts + clock,9.86,16.87
Counts + hub + clock,11.52,19.50
Counts + hub + history,11.04,19.37


Selected from validation: {5: 'E-bike count + hub', 10: 'E-bike count + hub'}


In [32]:
ten = means.xs(10, level="horizon")
names = list(dict.fromkeys(["Count baseline", selected[10], "Counts + hub + clock",
                            "Full snapshot + clock + history"]))
display(ten.loc[names, ["precision", "recall", "average_precision", "brier"]].round(4))
by_month = all_scores.query("horizon == 10").pivot(index="experiment", columns="month", values="precision")
display((100 * by_month.loc[names]).round(2))

,precision,recall,average_precision,brier
experiment,,,,
Count baseline,0.1460,0.4182,0.1403,0.0626
E-bike count + hub,0.1873,0.5339,0.1758,0.0622
Counts + hub + clock,0.1950,0.5561,0.1905,0.0619
Full snapshot + clock + history,0.1969,0.5605,0.2047,0.0612


month,2026-07,2026-08,2026-09
experiment,,,
Count baseline,12.41,13.70,17.68
E-bike count + hub,15.06,18.65,22.46
Counts + hub + clock,15.83,19.03,23.64
Full snapshot + clock + history,15.67,19.57,23.83


The richer model can score better on average precision even when it barely changes the top fifth of warnings. The rider-facing decision uses a fixed warning budget. Keep the model picked on that metric, and inspect each month so one unusually busy month cannot hide a reversal.

### Can the model tell two one-bike walks to the *same station* apart?

The campus-wide metric lets a model succeed by learning that one hub runs dry more often than another. A regular at Brock Commons wants to distinguish Tuesday at 08:00 from Saturday at noon at that same hub. For this diagnostic, use only one-bike walks and allocate each station its own 20% warning budget. A forecast that uses just hub and e-bike count must tie every walk within a station. Compare the original clock model with the campus-wide selection on validation.

In [33]:
names = list(dict.fromkeys(["Count baseline", selected[10], "Counts + hub + clock", "Context + history",
                            "Context + history + time at count"]))
within_station = means.xs(10, level="horizon").loc[names, ["precision", "same_hub_one_bike"]]
display((100 * within_station).round(2).rename(columns={"precision": "campus-wide precision (%)",
        "same_hub_one_bike": "one bike, budget per hub (%)"}))

,campus-wide precision (%),"one bike, budget per hub (%)"
experiment,,
Count baseline,14.60,14.60
E-bike count + hub,18.73,14.60
Counts + hub + clock,19.50,17.58
Context + history,19.68,19.44
Context + history + time at count,19.60,19.42


Holding the hub and starting count fixed changes the result. At ten minutes the clock model reaches 17.58% on this comparison, and adding recent history raises it to 19.44%. Apply the same simplicity rule to this rider-specific metric across every tested configuration. Carry both selections into the October replay and keep their objectives explicit.

In [34]:
local_precision = means.same_hub_one_bike.unstack().reindex(preference)
rider_choice = {h: local_precision.index[local_precision[h] >= local_precision[h].max() - TIE_MARGIN][0]
                for h in HORIZONS}
display(pd.DataFrame({"campus selection": selected, "same-hub one-bike selection": rider_choice,
                      "best within-hub precision (%)": (100 * local_precision.max()).round(2)}))
shortlist = list(dict.fromkeys([*selected.values(), *rider_choice.values(), "Counts + hub + clock"]))
display((100 * local_precision.loc[shortlist]).round(2))

,campus selection,same-hub one-bike selection,best within-hub precision (%)
5,E-bike count + hub,Counts + hub + history,11.98
10,E-bike count + hub,Tree · Counts + hub + clock,21.39


horizon,5,10
experiment,,
E-bike count + hub,8.47,14.60
Counts + hub + history,11.25,18.34
Tree · Counts + hub + clock,11.76,21.07
Counts + hub + clock,9.79,17.58


## 6. Replay the previously seen October benchmark

Refit both selected configurations on all eligible walks ending before October. Retain the first pilot's clock model as a reference, and compare both with the original count rule across all walks and again when exactly one bike shows. Each subgroup gets its own 20% warning budget. The three validation months informed the choice. October was already inspected during the original pilot, so this is a retrospective check.

In [35]:
train, october = forward_split(walks, TEST_MONTH, max(HORIZONS))
assert train.time.max() + pd.Timedelta(minutes=max(HORIZONS)) < october.time.min()
forecasts = {}
for h, name in selected.items():
    for candidate in dict.fromkeys(["Count baseline", name, rider_choice[h], "Counts + hub + clock"]):
        model, fields = specifications[candidate]
        forecasts[h, candidate] = predict(model, fields, train, october, h)
print(f"{len(october):,} October decisions, {october.station_id.nunique()} hubs, "
      f"{october.ebikes.eq(1).sum():,} decisions at one bike.")

18,947 October decisions, 17 hubs, 8,182 decisions at one bike.


In [36]:
def october_report(mask):
    rows = []
    for (h, name), scores in forecasts.items():
        y = october.loc[mask, f"empty_{h}"]
        row = metrics(y, scores[mask], WARNING_RATE)
        rows.append({"minutes": h, "model": name, **row})
    return pd.DataFrame(rows).set_index(["minutes", "model"]).round(4)

all_walks = np.ones(len(october), dtype=bool)
one_bike = october.ebikes.eq(1).to_numpy(dtype=bool)
october_report(all_walks)

walks  empty_arrivals  empty_rate  \
minutes model                                                            
5       Count baseline               18947             890      0.0470   
        E-bike count + hub           18947             890      0.0470   
        Counts + hub + history       18947             890      0.0470   
        Counts + hub + clock         18947             890      0.0470   
10      Count baseline               18947            1540      0.0813   
        E-bike count + hub           18947            1540      0.0813   
        Tree · Counts + hub + clock  18947            1540      0.0813   
        Counts + hub + clock         18947            1540      0.0813   

                                     precision  recall  average_precision  \
minutes model                                                               
5       Count baseline                  0.1002  0.4267             0.0976   
        E-bike count + hub              0.1231  0.5243             0.1293   
        Counts + hub + history          0.1243  0.5292             0.1361   
        Counts + hub + clock            0.1338  0.5697             0.1417   
10      Count baseline                  0.1654  0.4069             0.1591   
        E-bike count + hub              0.1972  0.4853             0.2035   
        Tree · Counts + hub + clock     0.2078  0.5112             0.2157   
        Counts + hub + clock            0.2095  0.5156             0.2082   

                                      brier  
minutes model                                
5       Count baseline               0.0433  
        E-bike count + hub           0.0430  
        Counts + hub + history       0.0427  
        Counts + hub + clock         0.0428  
10      Count baseline               0.0709  
        E-bike count + hub           0.0705  
        Tree · Counts + hub + clock  0.0700  
        Counts + hub + clock         0.0700

The one-bike subgroup holds the most immediately useful raw feature constant. A count-only rule has to break ties among these walks. Any improvement here comes from distinguishing walks with the same advertised e-bike count.

In [37]:
october_report(one_bike)

walks  empty_arrivals  empty_rate  \
minutes model                                                            
5       Count baseline                8182             820      0.1002   
        E-bike count + hub            8182             820      0.1002   
        Counts + hub + history        8182             820      0.1002   
        Counts + hub + clock          8182             820      0.1002   
10      Count baseline                8182            1353      0.1654   
        E-bike count + hub            8182            1353      0.1654   
        Tree · Counts + hub + clock   8182            1353      0.1654   
        Counts + hub + clock          8182            1353      0.1654   

                                     precision  recall  average_precision  \
minutes model                                                               
5       Count baseline                  0.1002  0.2000             0.1002   
        E-bike count + hub              0.1413  0.2819             0.1333   
        Counts + hub + history          0.1555  0.3102             0.1407   
        Counts + hub + clock            0.1573  0.3139             0.1467   
10      Count baseline                  0.1654  0.2000             0.1654   
        E-bike count + hub              0.2222  0.2687             0.2123   
        Tree · Counts + hub + clock     0.2523  0.3051             0.2258   
        Counts + hub + clock            0.2404  0.2908             0.2172   

                                      brier  
minutes model                                
5       Count baseline               0.0917  
        E-bike count + hub           0.0912  
        Counts + hub + history       0.0905  
        Counts + hub + clock         0.0907  
10      Count baseline               0.1417  
        E-bike count + hub           0.1409  
        Tree · Counts + hub + clock  0.1399  
        Counts + hub + clock         0.1399

Pooling all one-bike decisions still mixes easier and harder hubs. Repeat the validation diagnostic with a distinct warning budget at each of the 17 stations. This is the October comparison that matches the rider-specific selection metric.

In [38]:
one_hubs = october.loc[one_bike, "station_id"].to_numpy()
within_rows = []
for (h, name), scores_at_h in forecasts.items():
    observed = october.loc[one_bike, f"empty_{h}"].to_numpy(dtype=int)
    result = precision_within_stations(observed, scores_at_h[one_bike], one_hubs, WARNING_RATE)
    within_rows.append({"model": name, "minutes": h, "precision (%)": 100 * result})
display(pd.DataFrame(within_rows).pivot(index="model", columns="minutes", values="precision (%)").round(2))

minutes,5,10
model,,
Count baseline,10.02,16.54
Counts + hub + clock,12.70,20.91
Counts + hub + history,12.12,NaN
E-bike count + hub,10.02,16.54
Tree · Counts + hub + clock,NaN,21.93


### Is the campus-wide gain relevant at Brock Commons?

Inspect the home station on its own, restricting to one-bike walks. Each model gets the same warning budget within this station. The sample count and event count show how much evidence this narrower comparison has.

The preceding table leaves a cell empty when that configuration was selected only at the other walk duration.

In [39]:
brock_only = october.station_id.eq("2131").to_numpy(dtype=bool)
display(october_report(brock_only & one_bike)[["walks", "empty_arrivals", "precision", "recall"]])

walks  empty_arrivals  precision  recall
minutes model                                                                
5       Count baseline                 521              71     0.1363  0.2000
        E-bike count + hub             521              71     0.1363  0.2000
        Counts + hub + history         521              71     0.1631  0.2394
        Counts + hub + clock           521              71     0.2111  0.3099
10      Count baseline                 521             120     0.2303  0.2000
        E-bike count + hub             521             120     0.2303  0.2000
        Tree · Counts + hub + clock    521             120     0.3854  0.3346
        Counts + hub + clock           521             120     0.3647  0.3167

### How much does the estimate vary from day to day?

A single shortage can affect multiple consecutive walk starts. Resample whole Vancouver days with the competing forecasts paired. The intervals describe this month's variability at the given warning assignments. Consecutive days can still be correlated.

In [40]:
intervals = []
for h, name in selected.items():
    for label, mask in [("all", all_walks), ("one bike", one_bike)]:
        y = october.loc[mask, f"empty_{h}"]
        days = october.loc[mask, "time"].dt.tz_convert(TZ).dt.date
        candidate, baseline = forecasts[h, name][mask], forecasts[h, "Count baseline"][mask]
        lo, hi = paired_precision_interval(y, candidate, baseline, days.to_numpy(), WARNING_RATE)
        gain = metrics(y, candidate, WARNING_RATE)["precision"] - metrics(y, baseline, WARNING_RATE)["precision"]
        intervals.append([h, label, 100*gain, 100*lo, 100*hi])
display(pd.DataFrame(intervals, columns=["minutes", "walks", "gain (pp)", "lower 95%", "upper 95%"]).round(2))

,minutes,walks,gain (pp),lower 95%,upper 95%
0,5,all,2.29,1.72,2.93
1,5,one bike,4.10,2.57,5.67
2,10,all,3.19,2.30,4.14
3,10,one bike,5.68,3.47,8.01


For Brock Commons at exactly one e-bike, repeat the paired-day uncertainty check for the models that can change their advice from one visit to the next.

In [41]:
mask = brock_only & one_bike
brock_days = october.loc[mask, "time"].dt.tz_convert(TZ).dt.date.to_numpy()
rows = []
for name in dict.fromkeys([rider_choice[10], "Counts + hub + clock"]):
    y_brock = october.loc[mask, "empty_10"].to_numpy(dtype=int)
    candidate, baseline = forecasts[10, name][mask], forecasts[10, "Count baseline"][mask]
    bounds = paired_precision_interval(y_brock, candidate, baseline, brock_days, WARNING_RATE)
    gain = (metrics(y_brock, candidate, WARNING_RATE)["precision"]
            - metrics(y_brock, baseline, WARNING_RATE)["precision"])
    rows.append([name, 100*gain, 100*bounds[0], 100*bounds[1]])
display(pd.DataFrame(rows, columns=["model", "gain (pp)", "lower 95%", "upper 95%"]).round(2))

,model,gain (pp),lower 95%,upper 95%
0,Tree · Counts + hub + clock,15.50,3.69,27.17
1,Counts + hub + clock,13.44,3.78,23.10


### What does a warning mean in practice?

Look at the ten-minute score in risk bands and at the outcomes of warnings themselves. Tied scores at a boundary still share credit. The calibration table gives the average forecast beside the observed empty-arrival fraction.

In [42]:
scores = forecasts[10, selected[10]]
risk = pd.DataFrame({"forecast": scores, "empty": october.empty_10.to_numpy(dtype=int)})
risk["risk band"] = pd.qcut(risk.forecast, 5, labels=False, duplicates="drop") + 1  # 1 = lowest risk
display(risk.groupby("risk band", observed=True).agg(walks=("empty", "size"),
        mean_forecast=("forecast", "mean"), observed_empty=("empty", "mean")).round(3))
y = risk["empty"].to_numpy()
w = warning_weights(scores, WARNING_RATE)
display(pd.Series({"warnings per 100 walks": 100*w.mean(), "empty per 100 warnings": 100*np.average(y, weights=w),
                   "bike available per 100 warnings": 100*np.average(1-y, weights=w),
                   "empty arrivals missed per 100 empty arrivals": 100*np.average(1-w, weights=y)}).round(1).to_frame("10 min"))

,walks,mean_forecast,observed_empty
risk band,,,
1,3804,0.000,0.002
2,4297,0.008,0.010
3,3269,0.026,0.052
4,3858,0.086,0.152
5,3719,0.126,0.197


,10 min
warnings per 100 walks,20.0
empty per 100 warnings,19.7
bike available per 100 warnings,80.3
empty arrivals missed per 100 empty arrivals,51.5


### Could we use the warning rule before the rest of the month happens?

The fixed-budget comparison ranks an entire evaluation month. A phone app needs a cutoff in advance. Use the **September** ten-minute forecasts to set a score cutoff, refit the chosen model through September, then apply that fixed cutoff to October. Report the warning rate as well as its precision, since a shift between semesters can change the number of warnings riders see.

In [43]:
chosen_model, fields = specifications[selected[10]]
sep_train, sep = forward_split(walks, VALIDATION[-1], max(HORIZONS))
sep_scores = predict(chosen_model, fields, sep_train, sep, 10)
cutoff = np.quantile(sep_scores, 1 - WARNING_RATE, method="higher")
warn = scores >= cutoff   # warn all ties, as the simple live policy would
fixed = pd.Series({"September cutoff": cutoff, "September warned": (sep_scores >= cutoff).mean(),
                   "October warned": warn.mean(), "October warning precision": y[warn].mean() if warn.any() else np.nan,
                   "October recall": warn[y.astype(bool)].mean()})
display(fixed.round(4).to_frame("fixed cutoff, 10 min"))

,"fixed cutoff, 10 min"
September cutoff,0.0997
September warned,0.2052
October warned,0.2791
October warning precision,0.1884
October recall,0.6468


## What the pilot supports

**For a returning rider, the choice of comparison changes the model.** E-bike count and hub identity were enough to come within one percentage point of the best campus-wide validation precision. At one known hub with exactly one e-bike, both inputs are constant. That model assigns the same risk to every visit.

For five-minute walks in the same-hub, one-bike comparison, validation selected logistic regression with both counts, hub and recent inventory/report history. In October it caught 12.12 empty arrivals per 100 warnings across hubs, compared with 10.02 for count alone. The original clock model reached 12.70. For ten-minute walks the selected shallow tree used both counts, hub, local time and weekend. It reached **21.93 per 100 warnings**, compared with 16.54 for count alone.

**At Brock Commons**, 120 of 521 October one-bike walks had an empty ten-minute arrival. At the same warning budget, the count-only baseline caught 23.03 empty arrivals per 100 warnings, the clock logistic model 36.47, and the tree 38.54. Relative to count alone, the tree gained 15.50 percentage points, with a paired-day 95% interval from 3.69 to 27.17. That interval compares the tree with the baseline. The modest difference between the tree and the clock logistic model remains unresolved.

The campus model's September cutoff warned on 27.91% of October walks, above its intended 20%. Most warned walks still had a bike. A gentle backup suggestion fits the evidence better than telling a rider the bike will be gone. October was previously inspected and the rider-specific metric emerged through exploration. A new month with the feature sets, warning threshold and evaluation fixed in advance would provide an independent check. An actual unlock would require separate rider evidence.